### Setup

In [2]:
import sqlite3
import csv
from werkzeug.utils import secure_filename # For Exercise 3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()
print("Connection established for Additional Exercises.")

Connection established for Additional Exercises.


### 1. Course Enrollment Tracker

## Step 1: Create the Junction Table

In [3]:
# Create the enrollments table to link students to courses
cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
    student_id INTEGER,
    course_id TEXT,
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(id),
    FOREIGN KEY (course_id) REFERENCES courses(course_code)
)""")
conn.commit()
print("Junction table 'enrollments' created.") 

Junction table 'enrollments' created.


## Step 2: CLI Enrollment Program

In [5]:
def enroll_student():
    # 1. Input Student ID and Course ID [cite: 411, 413]
    s_id = int(input("Enter Student ID: "))
    c_id = input("Enter Course Code (e.g., BS ECE): ").upper()

    try:
        # 2. Insert into enrollments using parameterized query [cite: 417]
        cursor.execute("INSERT INTO enrollments (student_id, course_id) VALUES (?, ?)", (s_id, c_id))
        conn.commit()
        print(f"Success: Student {s_id} enrolled in {c_id}.")
    except sqlite3.IntegrityError:
        # 3. Check for duplicate entries [cite: 416, 425]
        print("Warning: This student is already enrolled in this course.")

enroll_student()

Success: Student 250082 enrolled in BS ECE.


### 2. Attendance Recording System

## Step 1: Create Attendance Table

In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    date TEXT DEFAULT (DATE('now')),
    status TEXT CHECK(status IN ('Present', 'Absent')),
    FOREIGN KEY (student_id) REFERENCES students(id)
)""")
conn.commit()

## Step 2: Mark Attendance

In [8]:
# Input details [cite: 433, 435]
s_id = int(input("Enter Student ID for attendance: "))
stat = input("Enter Status (Present/Absent): ").capitalize()

cursor.execute("INSERT INTO attendance (student_id, status) VALUES (?, ?)", (s_id, stat))
conn.commit()
print(f"Attendance recorded for Student {s_id}.")

Attendance recorded for Student 250082.


### 3. Profile Management with Image Upload

## Step 1: Update Students Table for Profiles

In [9]:
try:
    cursor.execute("ALTER TABLE students ADD COLUMN image_filename TEXT")
    conn.commit()
except sqlite3.OperationalError:
    print("Column already exists.")

Column already exists.


## Step 2: Profile Update Script

In [10]:
import os

# 1. Inputs [cite: 450, 451]
s_id = int(input("Enter Student ID to update: "))
new_name = input("Enter new display name: ")
filename = input("Enter image filename (e.g., profile.png): ")

# 2. Validation and Secure Filename [cite: 453, 454]
if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
    secure_name = secure_filename(filename) # Mimics Werkzeug utility [cite: 454]
    
    # 3. Update Database [cite: 456]
    cursor.execute("UPDATE students SET full_name = ?, image_filename = ? WHERE id = ?", 
                   (new_name, secure_name, s_id))
    conn.commit()
    print(f"Profile updated! Image '{secure_name}' linked to {new_name}.")
else:
    print("Invalid file format! Use .jpg, .png, or .jpeg.")

Profile updated! Image 'image.png' linked to Raym.
